### Zadania

Celem listy jest zbadanie działania metod binaryzacji obrazów w skali szarości i kolorowych. Dążymy do wyobrębnienia obiektów znajdujących się na podanych obrazach, tzn. uzyskania separacji obiektów od tła, a także oddzielenia poszczególnych obiektów. Natomiast samo osiągnięcie "jakiegoś" wyniku ([`cv2.threshold`](https://docs.opencv.org/3.4/d7/d1b/group__imgproc__misc.html#gae8a4a146d1ca78c626a53577199e9c57) odpalone, próg wybrany "z palca", _coś tam_ wychodzi, wynik wyświetlony) to co najwyżej połowa sukcesu i oceny. Ważne jest przeprowadzenie eksperymentów wyjaśniających dobór parametrów.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import binary_fill_holes

##### Zad. 1: Monety szare

Dane są dwa czarno-białe obrazy monet, `monety_szare` oraz `monety_szare_szum`, z których drugi zawiera pewien poziom zakłóceń. Dokonaj binaryzacji tych obrazów za pomocą progowania.  
Jak wartość progu wpływa na wyniki?  
Jaki wpływ mają zakłócenia w drugim obrazie? Zastosuj wybraną metodę odszumiania i przeanalizuj efekty.

In [ ]:
monety_szare = cv2.imread('Monety_szare.bmp')
monety_szum = cv2.imread('Monety_szare_szum.bmp')

In [ ]:
_, (a, b) = plt.subplots(1,2, figsize=(14,10))
a.imshow(monety_szare); a.title.set_text('1A: monety_szare')
b.imshow(monety_szum); b.title.set_text('1B: monety_szum')

Rozwiązanie części 1A:

In [ ]:
print("Rozmiar monety_szare: " + monety_szare.shape.__str__())
gray_monety_szare = cv2.cvtColor(monety_szare, cv2.COLOR_BGR2GRAY)

In [ ]:
plt.imshow(gray_monety_szare, cmap='gray')
a.title.set_text('1A: gray_monety_szare')

print("Rozmiar monety_szare: " + gray_monety_szare.shape.__str__())

**Konwertowanie do skali szarości (`cv2.COLOR_BGR2GRAY`)**
1. **Redukcja wymiarowości:** Obraz kolorowy ma 3 kanały (BGR). Konwersja sprowadza go do 1 kanału określającego jasność piksela (w zakresie 0–255).
2. **Wymóg algorytmu:** Klasyczne progowanie (`cv2.threshold`) działa na jednowymiarowej wartości jasności, a nie na wektorach kolorów.
3. **Wydajność i prostota:** Analiza 1 kanału jest znacznie szybsza obliczeniowo i pozwala na bezpośrednie wyznaczenie progu na podstawie jednolitego histogramu jasności.

In [ ]:
plt.hist(gray_monety_szare.flatten(), bins=256, color='gray', range=(0, 255))

plt.title('Histogram 1A: monety_szare')
plt.xlabel('Jasność piksela (0-255)')
plt.ylabel('Liczba pikseli')
plt.grid(True, linestyle='--', alpha=0.6)

progi = [95, 135, 165, 175, 200]
for p in progi:
    plt.axvline(x=p, color='red', linestyle='--', linewidth=1.5)

plt.show()

**Analiza histogramu i dobór progu binaryzacji:**
- **Struktura histogramu:** Histogram ma charakter bimodalny (widoczne dwa główne skupiska pikseli):
   * **Lewy szczyt (zakres ~50–100):** Odpowiada ciemniejszym pikselom tworzącym monety.
   * **Prawy szczyt (zakres ~150–200):** Odpowiada jasnemu tłu, na którym leżą monety.

In [ ]:
fig, axes = plt.subplots(1, len(progi), figsize=(18, 4))

for ax, p in zip(axes, progi):
    _, binary = cv2.threshold(gray_monety_szare, p, 255, cv2.THRESH_BINARY_INV)

    ax.imshow(binary, cmap='gray')
    ax.set_title(f'Próg T = {p}')

plt.tight_layout()
plt.show()

**Ocena wyznaczonych progów ($T$):**
   * **$T = 95$ (za niski):** Wcina się w szczyt monet, co powoduje odrzucenie ciemniejszych fragmentów samych monet (powstają dziury w obiektach).
   * **$T = 135$ (OPTYMALNY):** Leży dokładnie w dolinie (lokalnym minimum) pomiędzy szczytem monet a tłem. Zapewnia najdokładniejsze odseparowanie obiektów od tła.
   * **$T = 165$ oraz $T = 175$ (za wysoki):** Wchodzą w zakres pikseli tła, przez co cienie wokół monet oraz fragmenty tła zaczynają być błędnie klasyfikowane jako monety.
   * **$T = 200$ (bardzo za wysoki):** Przejmuje prawie całe tło obrazu.

### Wypełnienie białych środków

In [ ]:
_, binary = cv2.threshold(gray_monety_szare, 135, 255, cv2.THRESH_BINARY_INV)
filled_binary = binary_fill_holes(binary == 255).astype(np.uint8) * 255

plt.imshow(filled_binary, cmap='gray')
plt.title('Wypełnione dziury')
plt.show()

**Wypełnianie otworów (`binary_fill_holes`):** Ze względu na refleksy światła i wzory na monetach po binaryzacji powstają wewnątrz nich dziury. Operacja ta zamienia czarne piksele otoczone bielą na białe, co pozwala uzyskać pełną, jednolitą maskę obiektów do dalszej analizy.

Rozwiązanie części 1B:


In [ ]:
print("Rozmiar monety_szum: " + monety_szum.shape.__str__())
gray_monety_szum = cv2.cvtColor(monety_szum, cv2.COLOR_BGR2GRAY)

In [ ]:
print("Rozmiar monety_szum: " + str(gray_monety_szum.shape))

plt.imshow(gray_monety_szum, cmap='gray')
plt.title('1B: gray_monety_szum')
plt.show()

**Konwersja i ocena szumu (`gray_monety_szum`):** Po sprowadzeniu obrazu do jednokanałowej skali szarości widoczne jest silne, ziarniste zakłócenie na całym obszarze. Powoduje ono znaczne lokalne wahania jasności pikseli zarówno na monetach, jak i na tle, co utrudni poprawną binaryzację i wskazuje na konieczność wstępnego odszumienia obrazu.

In [ ]:
p_szum = 125

plt.hist(gray_monety_szum.flatten(), bins=256, color='gray', range=(0, 255))

plt.title('Histogram 1B: monety_szum')
plt.xlabel('Jasność piksela (0-255)')
plt.ylabel('Liczba pikseli')
plt.grid(True, linestyle='--', alpha=0.6)
plt.axvline(x=p_szum, color='red', linestyle='--', linewidth=1.5)

plt.tight_layout()
plt.show()

1. **Dlaczego na obrazie czystym było więcej pików?**
Na obrazie bez szumu widoczne były drobne pod-szczyty i detale. Wynikały one z subtelnych różnic w jasności poszczególnych monet (różne stopy metali, grawerunki, przetarcia) oraz cieni na tle.
2. **Dlaczego na obrazie z szumem są dwa wygładzone szczyty?**
Dodanie losowego szumu (o rozkładzie zbliżonym do Gaussa) nakłada na każdy piksel losowe odchylenie jasności. Szum ten „rozmywa” i uśrednia drobne detale tonalne, łącząc wąskie piki w dwa szerokie, wygładzone skupiska (rozkłady normalne) – jedno dla monet, drugie dla tła.
3. **Wpływ na wyznaczenie progu ($T = 125$):**
Próg $T = 125$ trafia w spłyconą dolinę między dwoma szczytami. Przez to, że rozkłady monet i tła mocno na siebie nachodzą, zwykłe progowanie bez wcześniejszej filtracji wygeneruje bardzo duży szum na obrazie binarnym.

In [ ]:
_, binary_szare = cv2.threshold(gray_monety_szare, p_szum, 255, cv2.THRESH_BINARY_INV)
_, binary_szum = cv2.threshold(gray_monety_szum, p_szum, 255, cv2.THRESH_BINARY_INV)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.imshow(binary_szare, cmap='gray')
ax1.set_title(f'Obraz bez szumu (T = {p_szum})')

ax2.imshow(binary_szum, cmap='gray')
ax2.set_title(f'Obraz z szumem (T = {p_szum})')

plt.tight_layout()
plt.show()

- Obraz bez szumu: Daje czytelną maskę monet o wyraźnych, gładkich krawędziach i czystym tle.

- Obraz z szumem: Bez wcześniejszej filtracji powstaje ogromna liczba artefaktów (tzw. szum pieprz i sól). Przypadkowe piksele tła przekraczają próg jasności (tworząc białe plamki na czarnym tle), a piksele monet opadają poniżej progu (tworząc czarne dziury wewnątrz obiektów).

In [ ]:
blurred_gray_monety_szum = cv2.GaussianBlur(gray_monety_szum, (7, 7), 0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.imshow(gray_monety_szum, cmap='gray')
ax1.set_title('Obraz z szumem')

ax2.imshow(blurred_gray_monety_szum, cmap='gray')
ax2.set_title('Obraz bez szumu')

plt.tight_layout()
plt.show()

- Zastosowanie filtra Gaussa skutecznie redukuje szum ziarnisty poprzez lokalne uśrednianie wartości pikseli z wagami gaussowskimi.
- Powierzchnie monet oraz tło stają się spójne tonalnie, co zapobiegnie powstawaniu pojedynczych, losowych artefaktów podczas późniejszej binaryzacji.
- Efektem ubocznym dolnoprzepustowej filtracji jest delikatne rozmycie ostrych krawędzi monet, jednak ogólna separacja tonalna obiektów od tła zostaje zachowana.

In [ ]:
p_szum = 135

plt.hist(blurred_gray_monety_szum.flatten(), bins=256, color='gray', range=(0, 255))

plt.title('Histogram 1B: monety_szum')
plt.xlabel('Jasność piksela (0-255)')
plt.ylabel('Liczba pikseli')
plt.grid(True, linestyle='--', alpha=0.6)
plt.axvline(x=p_szum, color='red', linestyle='--', linewidth=1.5)

plt.tight_layout()
plt.show()

- Filtracja wygładzająca wyeliminowała nakładające się na siebie ekstrema szumu, dzięki czemu spłycona wcześniej przestrzeń między szczytami zamieniła się w wyraźne, głębokie minimum tonalne.
* Histogram ponownie przypomina strukturę czystego obrazu `1A` – widoczne są dwa odseparowane skupiska (monety $\approx 70-110$, tło $\approx 180$).
* Czerwona linia progu $T = 135$ znajduje się idealnie w najniższym punkcie doliny. Pozwala to na optymalny podział pikseli na obiekty i tło, eliminując ryzyko powstawania szumu na obrazie binarnym.

In [ ]:
ksizes = [(3, 3), (7, 7), (9, 9), (13, 13)]
p = 135
fig1, axes1 = plt.subplots(1, len(ksizes), figsize=(18, 4))

blurred_images = []
for ax, ksize in zip(axes1, ksizes):
    blurred = cv2.GaussianBlur(gray_monety_szum, ksize, 0)
    blurred_images.append(blurred)

    ax.imshow(blurred, cmap='gray')
    ax.set_title(f'Filtr Gaussa: {ksize[0]}x{ksize[1]}')
    ax.axis('off')

plt.tight_layout()
plt.show()

fig2, axes2 = plt.subplots(1, len(ksizes), figsize=(18, 4))

for ax, blurred, ksize in zip(axes2, blurred_images, ksizes):
    _, binary = cv2.threshold(blurred, p, 255, cv2.THRESH_BINARY_INV)

    ax.imshow(binary, cmap='gray')
    ax.set_title(f'Binaryzacja T={p} ({ksize[0]}x{ksize[1]})')
    ax.axis('off')

plt.tight_layout()
plt.show()



1. **$3 \times 3$ (Niedostateczne odszumienie):** Małe jądro nie usuwa całości zakłóceń, przez co na obrazie binarnym wciąż pojawiają się drobiny szumu na tle oraz poszarpane wnętrza monet.


2. **$7 \times 7$ oraz $9 \times 9$ (Rozmiary optymalne):** Filtracja skutecznie eliminuje szum z tła, pozostawiając gładkie wnętrza monet przy jednoczesnym zachowaniu poprawnych krawędzi obiektów. Rozmiar $9 \times 9$ zapewnia najbardziej spójną i czystą maskę.


3. **$13 \times 13$ (Nadmierne rozmycie):** Tło jest całkowicie czyste, jednak zbyt duże jądro wywołuje silne rozmycie brzegów (zacieranie detali krawędziowych), co zniekształca obrysy monet.



* **Wniosek:** Dobór rozmiaru jądra jest kompromisem pomiędzy redukcją szumu a zachowaniem ostrości krawędzi obiektów. W rozważanym przypadku optymalny balans zapewnia jądro $7 \times 7$ lub $9 \times 9$.

In [ ]:
blurred = cv2.GaussianBlur(gray_monety_szum, (9, 9), 0)
_, binary = cv2.threshold(blurred, 135, 255, cv2.THRESH_BINARY_INV)
filled_binary = binary_fill_holes(binary == 255).astype(np.uint8) * 255

plt.imshow(filled_binary, cmap='gray')
plt.title('Wypełnione dziury')
plt.show()

##### Zad. 2: Monety kolorowe

Na podobnej zasadzie dokonaj binaryzacji poniższego obrazu kolorowego.  
Skorzystaj z modelu barw HSV. Uzasadnij dobór kanału oraz progu binaryzacji.

In [ ]:
monety_kolor = cv2.imread('Monety_kolor.bmp')
plt.subplots(1,1,figsize=(16,8))[1].imshow(monety_kolor[..., ::-1])
str(monety_kolor.shape)

* **Struktura i rozmiar:** Obraz ma wymiary $591 \times 1275$ pikseli i posiada 3 kanały barwne w formacie BGR (OpenCV).
* **Konwersja koloru do wyświetlenia:** Zastosowanie zapisu `monety_kolor[..., ::-1]` odwraca kolejność kanałów z BGR na RGB, co umożliwia ich prawidłowe wyrenderowanie przez bibliotekę Matplotlib.
* **Charakterystyka sceny:** Obraz zawiera monety o różnych odcieniach metali (srebrne, mosiężne, miedziane) położone na jednolicie czerwonym tle. Ze względu na dużą zmienność jasności samych monet, klasyczna konwersja do skali szarości utrudniłaby ich oddzielenie, co uzasadnia przejście do przestrzeni barw HSV.

In [ ]:
monety_kolor_hsv = cv2.cvtColor(monety_kolor, cv2.COLOR_BGR2HSV)
plt.imshow(monety_kolor_hsv[..., ::-1])

* **Cel konwersji:** Przejście z przestrzeni BGR do HSV rozdziela odcień barwy ($H$), jej nasycenie ($S$) oraz jasność ($V$), co umożliwia segmentację obiektów w oparciu o nasycenie lub kolor, a nie tylko o jasność.


* **Wygląd wizualizacji:** Jaskrawe, „fałszywe” kolory (np. jaskrawozielone tło) pojawiają się dlatego, że funkcja `plt.imshow` interpretuje wartości kanałów $H, S, V$ jako standardowe kanały $R, G, B$. Potwierdza to jednak poprawne przekształcenie macierzy przed jej rozdzieleniem na osobne kanały.

In [ ]:
h, s, v = cv2.split(monety_kolor_hsv)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(h, cmap='gray')
axes[0].set_title('Kanał H (Barwa)')

axes[1].imshow(s, cmap='gray')
axes[1].set_title('Kanał S (Nasycenie)')

axes[2].imshow(v, cmap='gray')
axes[2].set_title('Kanał V (Jasność)')

plt.tight_layout()
plt.show()

* **Kanał H (Barwa):** Określa dominujący odcień koloru. Dla metalicznych powierzchni monet wartość barwy jest niestabilna i szumowa, co uniemożliwia jednoznaczny podział.


* **Kanał S (Nasycenie) – KANAŁ OPTYMALNY:** Jaskrawo czerwone tło charakteryzuje się bardzo wysokim nasyceniem (jasne/białe piksele), podczas gdy monety z uwagi na metaliczny kolor mają nasycenie bliskie zeru (ciemne/czarne piksele). Tworzy to doskonały kontrast rzędu niemal $100\%$ między obiektami a tłem.
* **Kanał V (Jasność):** Odpowiada natężeniu światła. Występujące cienie i odbicia sprawiają, że jasność tła i monet na siebie nachodzi, co utrudnia progowanie.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Kanał H ma w OpenCV zakres 0-179
axes[0].hist(h.flatten(), bins=180, color='red', range=(0, 180))
axes[0].set_title('Histogram H (Barwa)')

# Kanały S i V mają zakres 0-255
axes[1].hist(s.flatten(), bins=256, color='green', range=(0, 255))
axes[1].set_title('Histogram S (Nasycenie)')

axes[2].hist(v.flatten(), bins=256, color='blue', range=(0, 255))
axes[2].set_title('Histogram V (Jasność)')

for ax in axes:
    ax.set_xlabel('Wartość piksela')
    ax.set_ylabel('Liczba pikseli')
    ax.grid(True, linestyle='--', alpha=0.5)

1. **Histogram H (Barwa):** Pokazuje dużą koncentrację wartości przy $0$ (czerwone tło), jednak odcienie monet są mocno rozproszone. Ze względu na metaliczny charakter monet barwa jest niestabilna i szumowa.
2. **Histogram S (Nasycenie) – IDEALNY DO BINARYZACJI:**
* Wykazuje doskonałą, podręcznikową **strukturyzację bimodalną**.
* **Lewy szczyt ($\approx 0-30$):** Odpowiada monetom, które nie mają nasyconego koloru (są acchromatyczne/szare).
* **Prawy szczyt ($\approx 240-255$):** Odpowiada jaskrawo czerwonemu tłu o bardzo wysokim nasyceniu.
* **Szeroka dolina ($\approx 50-200$):** Głęboki i płaski obszar zerowy między szczytami daje ogromny margines błędu przy doborze progu (np. $T = 100$ lub $T = 200$).


3. **Histogram V (Jasność):** Liczne, nakładające się na siebie piki wynikają z refleksów światła na monetach i nierównomiernego oświetlenia tła. Brak wyraźnej doliny wyklucza ten kanał z prostej binaryzacji globalnej.

In [ ]:
s_channel = monety_kolor_hsv[:, :, 1]

progi = [30, 100, 200]
fig, axes = plt.subplots(1, len(progi), figsize=(16, 5))

for ax, p in zip(axes, progi):
    _, binary = cv2.threshold(s_channel, p, 255, cv2.THRESH_BINARY_INV)

    ax.imshow(binary, cmap='gray')
    ax.set_title(f'Binarizacja S (Próg T = {p})')
    ax.axis('off')

plt.tight_layout()
plt.show()

* **Próg $T = 30$ (Za niski):** Część monet o cieplejszym odcieniu (mosiężnych i miedzianych) posiada niewielkie nasycenie barwy przekraczające 30. W efekcie są one segmentowane niepoprawnie, tworząc ubytki i czarne plamy.


* **Próg $T = 100$ (Średni):** Zapewnia lepszą separację dla większości monet, ale na powierzchni monet mosiężnych wciąż pozostają drobne czarne fragmenty.


* **Próg $T = 200$ (Optymalny):** Wykorzystuje szeroką dolinę z histogramu nasycenia. Ponieważ czerwone tło ma bardzo wysokie nasycenie ($\approx 240-255$), a wszystkie monety mają nasycenie poniżej 200, próg ten pozwala uzyskać pełne, pełnowymiarowe maski wszystkich obiektów bez ubytków.

In [ ]:
_, binary = cv2.threshold(s_channel, 200, 255, cv2.THRESH_BINARY_INV)
filled_binary = binary_fill_holes(binary == 255).astype(np.uint8) * 255

plt.imshow(filled_binary, cmap='gray')
plt.title('Wypełnione dziury')
plt.show()

Przetestuj opracowany powyżej algorytm na poniższym obrazie z wyraźnym zaszumieniem. Co się zmienia i dlaczego? Uzupełnij potok przetwarzania o odszumianie.

In [ ]:
monety_kolor_szum = cv2.imread('Monety_kolor_szum.bmp')
plt.subplots(1,1,figsize=(16,8))[1].imshow(monety_kolor_szum[..., ::-1])
monety_kolor_szum_hsv = cv2.cvtColor(monety_kolor_szum, cv2.COLOR_BGR2HSV)

In [ ]:
h, s, v = cv2.split(monety_kolor_szum_hsv)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(h, cmap='gray')
axes[0].set_title('Kanał H (Barwa)')

axes[1].imshow(s, cmap='gray')
axes[1].set_title('Kanał S (Nasycenie)')

axes[2].imshow(v, cmap='gray')
axes[2].set_title('Kanał V (Jasność)')

plt.tight_layout()
plt.show()

* **Kanał H (Barwa):** Szum kolorowy drastycznie zaburzył wartości odcienia barwy $H$. Tło i monety zlały się w szumową strukturę o zminimalizowanym kontraście.


* **Kanał S (Nasycenie):** Mimo widocznej ziarnistości szumu, kanał ten zachowuje **najwyższy kontrast** między achromatycznymi monetami (ciemny obszar) a nasyconym czerwonym tłem (jasny obszar). Pozostaje jedynym trafnym wyborem do segmentacji po uprzednim odszumieniu.


* **Kanał V (Jasność):** Liczne refleksy, cienie oraz nałożenie się poziomów jasności monet i tła uniemożliwiają skuteczną binaryzację na podstawie samego natężenia światła.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Kanał H ma w OpenCV zakres 0-179
axes[0].hist(h.flatten(), bins=180, color='red', range=(0, 180))
axes[0].set_title('Histogram H (Barwa)')

# Kanały S i V mają zakres 0-255
axes[1].hist(s.flatten(), bins=256, color='green', range=(0, 255))
axes[1].set_title('Histogram S (Nasycenie)')

axes[2].hist(v.flatten(), bins=256, color='blue', range=(0, 255))
axes[2].set_title('Histogram V (Jasność)')

for ax in axes:
    ax.set_xlabel('Wartość piksela')
    ax.set_ylabel('Liczba pikseli')
    ax.grid(True, linestyle='--', alpha=0.5)

* **Histogram H (Barwa):** Szum kolorowy wywołał silne rozproszenie wartości odcienia w całym zakresie $0-180$. Widoczne piki na krańcach ($0$ oraz $180$) wynikają ze specyfiki zawijania koloru czerwonego w przestrzeni HSV OpenCV, jednak całkowity brak czytelnej doliny uniemożliwia poprawną segmentację.


* **Histogram S (Nasycenie) – KANAŁ NAJBARDZIEJ ROBUSTNY:** Wciąż zachowuje strukturę bimodalną z dominującą szpilką w okolicy $S \approx 255$ (nasycone czerwone tło) oraz zakresem niższych wartości dla achromatycznych monet. Wpływ szumu spłycił dolinę między szczytami, co wyjaśnia powstanie błędnych pikseli przy bezpośredniej binaryzacji i potwierdza konieczność wygładzenia kanału przed progowaniem.


* **Histogram V (Jasność):** Wykazuje szeroki, ciągły rozkład jasności bez wyraźnego minimum. Podobnie jak na obrazie wolnym od szumu, składowa wartości jasności nie pozwala na rozdzielenie monet od tła.

In [ ]:
p_s = 200

s_clean = monety_kolor_hsv[:, :, 1]
s_szum = monety_kolor_szum_hsv[:, :, 1]

_, binary_clean = cv2.threshold(s_clean, p_s, 255, cv2.THRESH_BINARY_INV)
_, binary_szum = cv2.threshold(s_szum, p_s, 255, cv2.THRESH_BINARY_INV)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.imshow(binary_clean, cmap='gray')
ax1.set_title(f'Kanał S bez szumu (T = {p_s})')
ax1.axis('off')

ax2.imshow(binary_szum, cmap='gray')
ax2.set_title(f'Kanał S z szumem (T = {p_s})')
ax2.axis('off')

plt.tight_layout()
plt.show()

* **Obraz bez szumu:** Zapewnia czystą separację z całkowicie czarnym tłem oraz spójnymi maskami monet.


* **Obraz z szumem:** Szum kolorowy wywołuje silne wahania nasycenia w całym obszarze. Piksele tła, których nasycenie spada poniżej 200, tworzą gęsty szum ziarnisty na tle (białe punkty), natomiast na powierzchni monet powstają czarne ubytki i nieciągłości w miejscach, gdzie szum podbił nasycenie.


* **Wniosek:** Zakłócenia w przestrzeni HSV wykluczają poprawną bezpośrednią binaryzację bez wcześniejszego wygładzenia obrazu (np. filtrem Gaussa).

In [ ]:
ksizes = [(3, 3), (7, 7), (9, 9), (13, 13), (15, 15), (17, 17)]
p = 200
fig, axes = plt.subplots(len(ksizes), 2, figsize=(10, 18))

for (ax_blur, ax_bin), ksize in zip(axes, ksizes):
    blurred = cv2.GaussianBlur(s_szum, ksize, 0)
    ax_blur.imshow(blurred, cmap='gray')
    ax_blur.set_title(f'Filtr Gaussa: {ksize[0]}x{ksize[1]}')
    ax_blur.axis('off')

    _, binary = cv2.threshold(blurred, p, 255, cv2.THRESH_BINARY_INV)
    ax_bin.imshow(binary, cmap='gray')
    ax_bin.set_title(f'Binaryzacja T={p} ({ksize[0]}x{ksize[1]})')
    ax_bin.axis('off')

plt.tight_layout()
plt.show()

* **$3 \times 3$ (Za małe jądro):** Niedostateczna redukcja szumu ziarnistego – na tle pozostają białe artefakty, a wnętrza monet zawierają liczne czarne dziury.


* **$7 \times 7$ oraz $9 \times 9$ (Rozmiary optymalne):** Zapewniają najlepszy balans pomiędzy usunięciem zakłóceń a zachowaniem geometrii monet. Jądro $9 \times 9$ pozwala uzyskać całkowicie czyste tło i pełne maski bez zniekształcania krawędzi.


* **$13 \times 13$, $15 \times 15$, $17 \times 17$ (Zbyt duże jądra):** Zwiększanie rozmiaru filtra powyżej $9 \times 9$ nie poprawia już jakości tła, wywołuje natomiast nadmierne rozmycie krawędzi (ang. *edge blurring*). Skutkuje to zniekształceniem obrysów monet i sztucznym powiększeniem/zaokrągleniem ich mierzonych granic.


* **Podsumowanie:** Optymalnym rozmiarem jądra dla kanału $S$ przy progu $T = 200$ jest **$9 \times 9$**. Czy przechodzimy do kolejnej części notebooka dotyczącej lokalnej binaryzacji tekstu (Task 3)?

##### Zad. 3: Tekst

W tym przypadku należy wyodrębnić litery od bardzo nierównomiernie oświetlonego tła. Skorzystaj z binaryzacji metodą Otsu. Zastosuj binaryzację w sposób lokalny (oknem przesuwnym). Jaki wpływ ma rozmiar okna na jakość binaryzacji? Jak przesuwać okno?

_(Pełna punktacja za zadanie wymaga własnoręcznej implementacji okna przesuwnego - samo Otsu możesz wziąć z OpenCV. Użycie zamiast tego_ `cv2.adaptiveThreshold` _liczone będzie jako pół zadania, ale jedynie po demonstracji zrozumienia tej metody.)_

In [ ]:
tekst = cv2.imread('Tekst1.bmp')
gray_tekst = cv2.cvtColor(tekst, cv2.COLOR_BGR2GRAY)
plt.imshow(gray_tekst, cmap='gray')

Rozwiązanie:

In [ ]:
p_szum = 90

plt.hist(gray_tekst.flatten(), bins=256, color='gray', range=(0, 255))

plt.title('Histogram 3: tekst')
plt.xlabel('Jasność piksela (0-255)')
plt.ylabel('Liczba pikseli')
plt.grid(True, linestyle='--', alpha=0.6)
plt.axvline(x=p_szum, color='red', linestyle='--', linewidth=1.5)

plt.tight_layout()
plt.show()

* Histogram wykazuje dwa główne szczyty, jednak ich interpretacja różni się od standardowego obrazu bimodalnego.
* Pierwszy, wysoki szczyt ($\approx 45$) odpowiada nie tyle samemu tekstowi, co zacienionej części tła po prawej stronie obrazu.
* Drugi szczyt ($\approx 145$) odpowiada oświetlonym obszarom tła po lewej stronie.
* Tekst (czarne litery) z powodu niewielkiego udziału powierzchniowego zlewają się z tłem w całym zakresie jasności.
* Zaznaczona czerwona linia ($T = 90$) w dolinie histogramu pokazuje słabość binaryzacji globalnej: wyznaczenie jednego progu dla całego obrazu spowoduje, że zacienione tło po prawej stronie zostanie błędnie sklasyfikowane jako tekst, podczas gdy litery po lewej stronie mogą zostać całkowicie utracone.

In [ ]:
_, thresh_otsu = cv2.threshold(gray_tekst, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
plt.imshow(thresh_otsu, cmap='gray')
plt.title("Global Otsu Threshold")

* Globalny algorytm Otsu wyznacza jeden próg $T$ dla całego obrazu. Ze względu na przekątny gradient cienia, jasność czystego tła po prawej stronie jest niższa niż wyznaczony próg globalny, co skutkuje zaklasyfikowaniem całej prawej połowy dokumentu jako czarnej plamy.
* Cała treść w zacienionej części dokumentu została bezpowrotnie utracona, mimo że dla ludzkiego oka tekst w cieniu pozostaje czytelny.

In [ ]:
def local_otsu(gray_img: np.ndarray, window: int, min_std: float = 8.0) -> np.ndarray:
    h, w = gray_img.shape
    radius = window // 2

    padded_img = cv2.copyMakeBorder(gray_img, radius, radius, radius, radius, cv2.BORDER_REPLICATE)
    output = np.zeros((h, w), dtype=np.uint8)

    for row in range(h):
        for column in range(w):
            window_array = padded_img[row : row + window, column : column + window]
            std_dev = np.std(window_array)

            if std_dev < min_std:
                output[row, column] = 255
            else:
                otsu_thresh, _ = cv2.threshold(window_array, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

                if gray_img[row, column] > otsu_thresh:
                    output[row, column] = 255
                else:
                    output[row, column] = 0

    return output

In [ ]:
windows = [15, 31, 51, 101]

fig, axes = plt.subplots(3, 2, figsize=(12, 15), dpi=120)
axes = axes.ravel()

axes[0].imshow(gray_tekst, cmap='gray')
axes[0].set_title('Oryginał (Grayscale)')
axes[0].axis('off')

global_thresh, global_bin = cv2.threshold(gray_tekst, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
axes[1].imshow(global_bin, cmap='gray')
axes[1].set_title(f'Globalne Otsu (T={int(global_thresh)})')
axes[1].axis('off')

for idx, win in enumerate(windows, start=2):
    result = local_otsu(gray_tekst, win)
    axes[idx].imshow(result, cmap='gray')
    axes[idx].set_title(f'Lokalne Otsu ({win}x{win})')
    axes[idx].axis('off')

plt.tight_layout(pad=1.5)
plt.subplots_adjust(wspace=0.1, hspace=0.25)
plt.show()

### 1. Wpływ rozmiaru okna ($W \times W$) na jakość binaryzacji

* **Globalne Otsu ($T = 89$):** Całkowicie uniemożliwia odczytanie prawej strony dokumentu ze względu na obecność silnego gradientu oświetlenia. Jednolity próg wyznaczony dla całego obrazu uznaje zacienione tło za obiekt (czarne piksele).


* **Małe okno ($15 \times 15$):** Pozwala wyeliminować wpływ cienia, jednak rozmiar okna jest porównywalny z grubością liter i odstępami między nimi. W obszarach o bardzo słabym kontraście krawędzie znaków mogą ulegać nieznacznemu postrzępieniu.


* **Optymalne okno ($31 \times 31$ oraz $51 \times 51$):** Zapewniają **najwyższą jakość segmentacji**. Okno jest wystarczająco duże, aby w jego obrębie niemal zawsze znajdował się zarówno fragment tła, jak i tekstu (co daje lokalny bimodalny histogram dla algorytmu Otsu), a zarazem wystarczająco małe, by lokalny gradient oświetlenia był pomijalny.


* **Duże okno ($101 \times 101$):** Zaczyna ujawniać wady zbliżone do binaryzacji globalnej. Na obszarach o gwałtownym spadku jasności (prawa górna krawędź) okno obejmuje zbyt duży zakres rozpiętości tonalnej, co powoduje ponowne pojawianie się artefaktów cienia.


### 2. Rola warunku odchylenia standardowego (`min_std`)

Zastosowanie progu variance/odchylenia standardowego (`std_dev < min_std`) jest kluczowym elementem poprawnej lokalnej binaryzacji Otsu:

* W obszarach **jednorodnego tła** (gdzie nie ma tekstu) lokalny histogram jest jednopolowy (unimodalny).
* Klasyczny algorytm Otsu wyznaczyłby w takim oknie próg w połowie szumu, dzieląc jednolite tło na czarne i białe piksele (powstanie szumów/artefaktów).
* Warunek `std_dev < 8.0` wymusza zaklasyfikowanie całego jednorodnego okna jako tła (wartość 255), eliminując fałszywe detekcje na czystym papierze.


### 3. Sposób przesuwania okna (Odpowiedź na pytanie z treści zadania)

W zaprezentowanej implementacji zastosowano przesuwanie **piksel po pikselu** (`stride = 1`) z wykorzystaniem nakładania ramki (`cv2.copyMakeBorder`):

* **Przesuwanie pikselowe (`stride = 1`):** Wyznacza próg lokalny indywidualnie dla otoczenia każdego piksela. Daje najbardziej płynne i precyzyjne rezultaty bez efektu blokowego (siatki) na granicach okien, choć wymaga większej mocy obliczeniowej.
* **Przesuwanie blokowe (krok równy rozmiarowi okna):** Obraz dzielony jest na rozłączne kafelki. Jest znaczne szybsze, ale może powodować widoczne skoki wartości progowych na krawędziach sąsiednich bloków.